# Teradata Vantage - Python Data Analysis Demo

This notebook demonstrates:
- Connecting to Teradata Vantage using Python
- Executing SQL queries and loading results into pandas DataFrames
- Data analysis and visualization
- Working with the `${DB_USERNAME}_DEMO` database

**Prerequisites**: Tables must be created first (run `00-setup.sql` or `00-setup.bteq`)

## 1. Setup and Connection

Import required libraries and establish connection to Teradata.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import teradatasql
from datetime import datetime

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

In [ ]:
# Get connection parameters from environment variables
db_host = os.environ.get('DB_HOST')
db_user = os.environ.get('DB_USERNAME')
db_pass = os.environ.get('DB_PASSWORD')
db_name = f"{db_user}_DEMO"

print(f"Connecting to: {db_host}")
print(f"Database: {db_name}")
print(f"User: {db_user}")

In [ ]:
# Establish connection to Teradata
connection = teradatasql.connect(
    host=db_host,
    user=db_user,
    password=db_pass,
    database=db_name
)

print("✓ Connected to Teradata Vantage successfully!")

## 2. Helper Functions

Create utility functions for executing queries and displaying results.

In [ ]:
def execute_query(sql, description=None):
    """
    Execute SQL query and return results as pandas DataFrame.
    
    Args:
        sql: SQL query string
        description: Optional description to print
    
    Returns:
        pandas DataFrame with query results
    """
    if description:
        print(f"\n{'='*60}")
        print(f"📊 {description}")
        print('='*60)
    
    df = pd.read_sql(sql, connection)
    return df

def display_table_info(table_name):
    """
    Display information about a table: row count and sample data.
    
    Args:
        table_name: Name of the table
    """
    # Get row count
    count_sql = f"SELECT COUNT(*) as row_count FROM {db_name}.{table_name}"
    count = execute_query(count_sql).iloc[0]['row_count']
    
    print(f"\n📋 Table: {table_name}")
    print(f"   Rows: {count:,}")
    
    # Get sample data
    sample_sql = f"SELECT * FROM {db_name}.{table_name} SAMPLE 5"
    sample_df = execute_query(sample_sql)
    
    print("\n   Sample data:")
    display(sample_df)
    
    return count

print("✓ Helper functions defined")

## 3. Data Exploration

Explore the available tables and their contents.

In [ ]:
# List all tables in the database
tables_sql = f"""
SELECT TableName, CreateTimeStamp, LastAlterTimeStamp 
FROM DBC.TablesV 
WHERE DatabaseName = '{db_name}'
  AND TableKind = 'T'
ORDER BY TableName
"""

tables_df = execute_query(tables_sql, "Available Tables")
display(tables_df)

In [ ]:
# Display information about each table
for table in ['DEPARTMENT', 'EMPLOYEE', 'PRODUCT', 'SALES']:
    try:
        display_table_info(table)
    except Exception as e:
        print(f"⚠️  Table {table} not found. Run setup scripts first.")

## 4. Basic Queries and Analysis

Execute queries similar to the SQL exercise files.

In [ ]:
# Employee analysis
employees_sql = """
SELECT 
    e.EMP_ID,
    e.FIRST_NAME,
    e.LAST_NAME,
    e.JOB_TITLE,
    e.SALARY,
    e.HIRE_DATE,
    d.DEPT_NAME,
    d.LOCATION
FROM EMPLOYEE e
INNER JOIN DEPARTMENT d ON e.DEPT_ID = d.DEPT_ID
ORDER BY e.SALARY DESC
"""

employees_df = execute_query(employees_sql, "Employee Directory with Departments")
display(employees_df)

In [ ]:
# Department statistics
dept_stats_sql = """
SELECT 
    d.DEPT_NAME,
    d.LOCATION,
    d.BUDGET,
    COUNT(e.EMP_ID) as EMPLOYEE_COUNT,
    AVG(e.SALARY) as AVG_SALARY,
    MIN(e.SALARY) as MIN_SALARY,
    MAX(e.SALARY) as MAX_SALARY,
    SUM(e.SALARY) as TOTAL_PAYROLL
FROM DEPARTMENT d
LEFT JOIN EMPLOYEE e ON d.DEPT_ID = e.DEPT_ID
GROUP BY d.DEPT_NAME, d.LOCATION, d.BUDGET
ORDER BY EMPLOYEE_COUNT DESC
"""

dept_stats_df = execute_query(dept_stats_sql, "Department Statistics")
display(dept_stats_df)

In [ ]:
# Sales analysis
sales_sql = """
SELECT 
    s.SALE_DATE,
    s.REGION,
    p.PRODUCT_NAME,
    p.CATEGORY,
    s.QUANTITY,
    s.SALE_AMOUNT,
    s.SALE_AMOUNT / s.QUANTITY as UNIT_PRICE
FROM SALES s
INNER JOIN PRODUCT p ON s.PRODUCT_ID = p.PRODUCT_ID
ORDER BY s.SALE_DATE DESC
"""

sales_df = execute_query(sales_sql, "Sales Transactions")
display(sales_df.head(10))

## 5. Data Visualization

Create visualizations using matplotlib and seaborn.

In [ ]:
# Visualization 1: Salary distribution by department
fig, ax = plt.subplots(figsize=(12, 6))

dept_order = dept_stats_df.sort_values('AVG_SALARY', ascending=False)['DEPT_NAME']
sns.barplot(data=dept_stats_df, x='DEPT_NAME', y='AVG_SALARY', order=dept_order, palette='viridis', ax=ax)

ax.set_title('Average Salary by Department', fontsize=16, fontweight='bold')
ax.set_xlabel('Department', fontsize=12)
ax.set_ylabel('Average Salary ($)', fontsize=12)
ax.tick_params(axis='x', rotation=45)

# Add value labels on bars
for i, v in enumerate(dept_stats_df.sort_values('AVG_SALARY', ascending=False)['AVG_SALARY']):
    ax.text(i, v + 2000, f'${v:,.0f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Employee count and total payroll by department
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Employee count pie chart
colors = sns.color_palette('Set3', len(dept_stats_df))
ax1.pie(dept_stats_df['EMPLOYEE_COUNT'], 
        labels=dept_stats_df['DEPT_NAME'], 
        autopct='%1.1f%%',
        colors=colors,
        startangle=90)
ax1.set_title('Employee Distribution by Department', fontsize=14, fontweight='bold')

# Total payroll bar chart
sns.barplot(data=dept_stats_df, x='DEPT_NAME', y='TOTAL_PAYROLL', palette='muted', ax=ax2)
ax2.set_title('Total Payroll by Department', fontsize=14, fontweight='bold')
ax2.set_xlabel('Department', fontsize=12)
ax2.set_ylabel('Total Payroll ($)', fontsize=12)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: Sales by region and category
sales_summary_sql = """
SELECT 
    s.REGION,
    p.CATEGORY,
    COUNT(*) as TRANSACTION_COUNT,
    SUM(s.QUANTITY) as TOTAL_QUANTITY,
    SUM(s.SALE_AMOUNT) as TOTAL_REVENUE
FROM SALES s
INNER JOIN PRODUCT p ON s.PRODUCT_ID = p.PRODUCT_ID
GROUP BY s.REGION, p.CATEGORY
ORDER BY TOTAL_REVENUE DESC
"""

sales_summary_df = execute_query(sales_summary_sql)

# Create pivot table for heatmap
sales_pivot = sales_summary_df.pivot(index='CATEGORY', columns='REGION', values='TOTAL_REVENUE')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(sales_pivot, annot=True, fmt='.0f', cmap='YlGnBu', cbar_kws={'label': 'Revenue ($)'}, ax=ax)
ax.set_title('Revenue Heatmap: Category vs Region', fontsize=16, fontweight='bold')
ax.set_xlabel('Region', fontsize=12)
ax.set_ylabel('Category', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📊 Sales Summary by Region and Category:")
display(sales_summary_df)

In [ ]:
# Visualization 4: Time series - Sales over time
sales_by_date_sql = """
SELECT 
    SALE_DATE,
    REGION,
    SUM(SALE_AMOUNT) as DAILY_REVENUE
FROM SALES
GROUP BY SALE_DATE, REGION
ORDER BY SALE_DATE, REGION
"""

sales_by_date_df = execute_query(sales_by_date_sql)
sales_by_date_df['SALE_DATE'] = pd.to_datetime(sales_by_date_df['SALE_DATE'])

fig, ax = plt.subplots(figsize=(14, 6))

for region in sales_by_date_df['REGION'].unique():
    region_data = sales_by_date_df[sales_by_date_df['REGION'] == region]
    ax.plot(region_data['SALE_DATE'], region_data['DAILY_REVENUE'], marker='o', label=region, linewidth=2)

ax.set_title('Daily Revenue by Region', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Revenue ($)', fontsize=12)
ax.legend(title='Region', loc='best')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Advanced Analytics

Perform more complex analytical queries.

In [ ]:
# Window functions - Rank employees by salary within each department
ranking_sql = """
SELECT 
    e.FIRST_NAME || ' ' || e.LAST_NAME as EMPLOYEE_NAME,
    d.DEPT_NAME,
    e.SALARY,
    RANK() OVER (PARTITION BY d.DEPT_NAME ORDER BY e.SALARY DESC) as SALARY_RANK,
    AVG(e.SALARY) OVER (PARTITION BY d.DEPT_NAME) as DEPT_AVG_SALARY
FROM EMPLOYEE e
INNER JOIN DEPARTMENT d ON e.DEPT_ID = d.DEPT_ID
QUALIFY SALARY_RANK <= 3
ORDER BY d.DEPT_NAME, SALARY_RANK
"""

ranking_df = execute_query(ranking_sql, "Top 3 Earners by Department")
display(ranking_df)

In [ ]:
# Product performance analysis
product_perf_sql = """
SELECT 
    p.PRODUCT_NAME,
    p.CATEGORY,
    p.UNIT_PRICE as LIST_PRICE,
    COUNT(s.SALE_ID) as TIMES_SOLD,
    SUM(s.QUANTITY) as TOTAL_UNITS_SOLD,
    SUM(s.SALE_AMOUNT) as TOTAL_REVENUE,
    AVG(s.SALE_AMOUNT / s.QUANTITY) as AVG_SELLING_PRICE
FROM PRODUCT p
LEFT JOIN SALES s ON p.PRODUCT_ID = s.PRODUCT_ID
GROUP BY p.PRODUCT_NAME, p.CATEGORY, p.UNIT_PRICE
HAVING SUM(s.SALE_AMOUNT) IS NOT NULL
ORDER BY TOTAL_REVENUE DESC
"""

product_perf_df = execute_query(product_perf_sql, "Product Performance Analysis")
display(product_perf_df)

In [ ]:
# Top products visualization
fig, ax = plt.subplots(figsize=(12, 8))

top_products = product_perf_df.head(10)
sns.barplot(data=top_products, y='PRODUCT_NAME', x='TOTAL_REVENUE', palette='rocket', ax=ax)

ax.set_title('Top 10 Products by Revenue', fontsize=16, fontweight='bold')
ax.set_xlabel('Total Revenue ($)', fontsize=12)
ax.set_ylabel('Product', fontsize=12)

plt.tight_layout()
plt.show()

## 7. Summary Statistics

Generate overall summary statistics using pandas.

In [ ]:
# Employee salary statistics
print("📊 Employee Salary Statistics")
print("="*60)
print(employees_df['SALARY'].describe())
print(f"\nTotal Payroll: ${employees_df['SALARY'].sum():,.2f}")

In [ ]:
# Sales statistics
print("📊 Sales Statistics")
print("="*60)
print(sales_df['SALE_AMOUNT'].describe())
print(f"\nTotal Revenue: ${sales_df['SALE_AMOUNT'].sum():,.2f}")
print(f"Total Transactions: {len(sales_df):,}")
print(f"Average Transaction Value: ${sales_df['SALE_AMOUNT'].mean():,.2f}")

In [ ]:
# Region performance comparison
region_stats = sales_df.groupby('REGION').agg({
    'SALE_AMOUNT': ['count', 'sum', 'mean', 'max'],
    'QUANTITY': 'sum'
}).round(2)

region_stats.columns = ['Transactions', 'Total_Revenue', 'Avg_Revenue', 'Max_Revenue', 'Total_Quantity']

print("\n📊 Regional Performance:")
display(region_stats)

## 8. Custom Analysis

Space for your own queries and analysis.

In [ ]:
# Write your own SQL query here
custom_sql = """
-- Add your custom SQL query here
SELECT 'Replace this with your query' as MESSAGE
"""

# Uncomment to execute:
# custom_df = execute_query(custom_sql, "Custom Analysis")
# display(custom_df)

## 9. Cleanup

Close the database connection when finished.

In [ ]:
# Close connection
connection.close()
print("✓ Connection closed successfully")

---

## Next Steps

1. **Explore More Data**: Try different SQL queries in Section 8
2. **Create Custom Visualizations**: Use matplotlib/seaborn to create your own charts
3. **Advanced Analytics**: Experiment with window functions, CTEs, and aggregations
4. **Machine Learning**: Use scikit-learn with this data for predictive modeling
5. **Export Data**: Save results to CSV using `df.to_csv('filename.csv')`

---

**Resources:**
- [Teradata Python Driver Documentation](https://github.com/Teradata/python-driver)
- [Pandas Documentation](https://pandas.pydata.org/docs/)
- [Matplotlib Gallery](https://matplotlib.org/stable/gallery/index.html)
- [Seaborn Tutorial](https://seaborn.pydata.org/tutorial.html)